# Interfaces Abstractas (`IList` e `ISet`)

In [1]:
#ifndef ISET_HPP
#define ISET_HPP

#include <cstddef> // Necesario para std::size_t

/**
 * @brief Interfaz abstracta para un conjunto (Set).
 *
 * Define las operaciones básicas que cualquier implementación de un conjunto 
 * debe proveer. Utiliza plantillas (templates) para soportar cualquier tipo 
 * de dato genérico T.
 *
 * @tparam T Tipo de los elementos que almacenará el conjunto.
 */
template <typename T>
class ISet {
protected:
    /** * @brief Cantidad actual de elementos en el conjunto.
     * Marcado como 'protected' (según el diagrama con '#') para que las clases 
     * derivadas puedan acceder a él y actualizarlo.
     */
    std::size_t size = 0;

public:
    /**
     * @brief Destructor virtual por defecto.
     * Regla de oro en C++: toda clase con métodos virtuales DEBE tener un destructor virtual.
     */
    virtual ~ISet() = default;

    /**
     * @brief Obtiene la cantidad de elementos en el conjunto.
     * @return std::size_t La cantidad de elementos.
     */
    virtual std::size_t getSize() const {
        return size;
    }

    /**
     * @brief Verifica si el conjunto está vacío.
     * @return true si el conjunto no tiene elementos, false en caso contrario.
     */
    virtual bool isEmpty() const {
        return size == 0;
    }

    // ========================================================================
    // Métodos puramente virtuales a ser implementados por clases derivadas
    // ========================================================================

    /**
     * @brief Elimina todos los elementos del conjunto.
     */
    virtual void clear() = 0;

    /**
     * @brief Imprime los elementos del conjunto en un flujo de salida.
     */
    virtual void print() const = 0;

    /**
     * @brief Verifica si un elemento específico pertenece al conjunto.
     * @param item Elemento a buscar.
     * @return true si el elemento está en el conjunto, false en caso contrario.
     */
    virtual bool contains(const T& item) const = 0;

    /**
     * @brief Añade un elemento al conjunto.
     * @param item Elemento a añadir.
     * @return true si se añadió con éxito, false si el elemento ya existía.
     */
    virtual bool add(const T& item) = 0;

    /**
     * @brief Elimina un elemento del conjunto.
     * @param item Elemento a eliminar.
     * @return true si se eliminó con éxito, false si el elemento no se encontró.
     */
    virtual bool remove(const T& item) = 0;
};

#endif // ISET_HPP

## Descripción

* ***Uso de Templates (`template <typename T>`)***: Dado que la clase debe ser genérica (como se observa en las dependencias ~`T`~ del diagrama), usamos plantillas. Esto permite que el ISet funcione con enteros, cadenas de texto u objetos personalizados en tiempo de compilación.

* ***Atributo protected***: El diagrama especifica `# size_t size`. En UML, el prefijo `#` indica visibilidad protegida. Esto significa que las clases que implementen ISet (como `ChainedHashTable` y `LinearHashTable`) heredarán la variable y podrán modificarla directamente sin necesidad de llamar a un setter, lo cual optimiza y facilita el diseño interno de las estructuras.

* ***Implementación en la Interfaz (`getSize` y `isEmpty`):*** Aunque es una "interfaz" (en C++ esto se traduce en una clase base abstracta), el estándar permite implementar métodos concretos. Como `getSize()` y `isEmpty()` dependen directamente de la variable size y su lógica será idéntica para cualquier implementación derivada, implementarlos aquí evita duplicar código en el futuro.

* ***Destructor Virtual (`virtual ~ISet() = default;`):*** En C++17, es crítico declarar el destructor como virtual en las clases base. Si no se hace, al eliminar una clase derivada a través de un puntero de la clase base, no se llamará al destructor de la clase derivada, causando fugas de memoria (memory leaks).

* ***Uso de const (Const-correctness):*** Métodos como `getSize()`, `isEmpty()`, `print()` y `contains()` llevan el modificador de acceso `const` al final porque no alteran el estado interno de la clase.

* ***Los parámetros pasados por valor en el UML (`item: T`)*** han sido optimizados pasándolos como referencias constantes (const T& item). Esto es una buena práctica en C++ para evitar copias innecesarias en memoria de objetos pesados, garantizando al mismo tiempo que el método no alterará el objeto original.

* ***Métodos Puramente Virtuales (`= 0`)***: Las operaciones específicas como `add`, `remove` o `contains` están igualadas a cero, haciendo que la clase sea puramente abstracta. Las clases derivadas estarán obligadas a proveer el código de estos métodos.

## El macro de guarda 
```
#ifndef ISET_HPP  // 1. Si NO está definido 'ISET_HPP'...
#define ISET_HPP  // 2. ...entonces defínelo.

// ... todo el código de la clase ...

#endif            // 3. Fin de la condición.
```

1. ***El problema de la doble inclusión***: En C++, cuando usas la directiva `#include "ISet.hpp"`, el preprocesador del compilador literalmente "copia y pega" todo el contenido de ese archivo de cabecera (header) en tu archivo .cpp. Imagina este escenario muy común en proyectos grandes:
    * Tienes una clase A que hace `#include "ISet.hpp"`.
    * Tienes una clase B que también hace `#include "ISet.hpp"`.

    Finalmente, tienes un archivo main.cpp que incluye tanto a la clase A como a la clase B.

    Si no tuviéramos las guardas de inclusión, el preprocesador pegaría el código de `ISet` dos veces dentro de `main.cpp`. Esto violaría una regla estricta de C++ llamada ODR (One Definition Rule), la cual dicta que una clase solo puede definirse una única vez. El compilador arrojaría un error de redefinición de clase y fallaría.

2. La solución se tiene a través de `#ifndef`. Las guardas de inclusión evitan este error actuando como un semáforo:
   * ***La primera vez que se incluye el archivo***: El compilador llega a `#ifndef ISET_HPP` (*If Not Defined* / Si no está definido). Como es la primera vez, la variable o macro `ISET_HPP` no existe. Así que entra, lee `#define ISET_HPP` (creando la macro en la memoria del compilador) y continúa leyendo la clase.

    * ***La segunda vez que se incluye el archivo***: Si otro archivo intenta volver a incluir `ISet.hpp` en el mismo proceso de compilación, el compilador llega de nuevo a `#ifndef ISET_HPP`. Pero esta vez, ya está definido (lo definimos en el paso anterior). Por lo tanto, el preprocesador salta automáticamente todo el código hasta encontrar el `#endif`, ignorando la clase entera y evitando el error de código duplicado.

3. El nombre `ISET_HPP`
El nombre `ISET_HPP` es arbitrario, pero la convención dicta que debe ser un identificador único en todo tu proyecto. Generalmente, se usa el nombre del archivo en mayúsculas, reemplazando el punto de la extensión por un guion bajo (`_`).

In [ ]:
#ifndef ILIST_HPP
#define ILIST_HPP

#include <cstddef> // Necesario para std::size_t

/**
 * @brief Interfaz abstracta para una Lista (List).
 *
 * Define las operaciones posicionales y secuenciales que cualquier 
 * implementación de lista (como arreglos dinámicos o listas enlazadas) 
 * debe proveer.
 *
 * @tparam T Tipo de los elementos que almacenará la lista.
 */
template <typename T>
class IList {
protected:
    /** * @brief Cantidad actual de elementos en la lista.
     * Protegido para el acceso directo de las clases derivadas.
     */
    std::size_t size = 0;

public:
    /**
     * @brief Destructor virtual por defecto.
     */
    virtual ~IList() = default;

    /**
     * @brief Obtiene la cantidad de elementos en la lista.
     * @return std::size_t La cantidad de elementos.
     */
    virtual std::size_t getSize() const {
        return size;
    }

    /**
     * @brief Verifica si la lista está vacía.
     * @return true si la lista no tiene elementos, false en caso contrario.
     */
    virtual bool isEmpty() const {
        return size == 0;
    }

    // ========================================================================
    // Métodos puramente virtuales a ser implementados por clases derivadas
    // ========================================================================

    /**
     * @brief Elimina todos los elementos de la lista.
     */
    virtual void clear() = 0;

    /**
     * @brief Imprime los elementos de la lista en orden.
     */
    virtual void print() const = 0;

    /**
     * @brief Obtiene el elemento en una posición específica.
     * @param index Posición del elemento a obtener (basado en 0).
     * @return T Una copia del elemento en la posición dada.
     */
    virtual T get(std::size_t index) const = 0;

    /**
     * @brief Reemplaza el elemento en una posición específica.
     * @param index Posición del elemento a modificar.
     * @param item Nuevo elemento que reemplazará al existente.
     */
    virtual void set(std::size_t index, const T& item) = 0;

    /**
     * @brief Busca la primera aparición de un elemento en la lista.
     * @param item Elemento a buscar.
     * @return int El índice del elemento si se encuentra, o -1 si no existe.
     */
    virtual int indexOf(const T& item) const = 0;

    /**
     * @brief Inserta un elemento en una posición específica, desplazando los demás.
     * @param index Posición donde se insertará el elemento.
     * @param item Elemento a insertar.
     */
    virtual void insertAt(std::size_t index, const T& item) = 0;

    /**
     * @brief Elimina el elemento en una posición específica.
     * @param index Posición del elemento a eliminar.
     */
    virtual void removeAt(std::size_t index) = 0;

    /**
     * @brief Inserta un elemento al principio de la lista.
     * @param item Elemento a insertar.
     */
    virtual void pushFront(const T& item) = 0;

    /**
     * @brief Inserta un elemento al final de la lista.
     * @param item Elemento a insertar.
     */
    virtual void pushBack(const T& item) = 0;

    /**
     * @brief Elimina y retorna el primer elemento de la lista.
     * @return T El elemento que fue eliminado.
     */
    virtual T popFront() = 0;

    /**
     * @brief Elimina y retorna el último elemento de la lista.
     * @return T El elemento que fue eliminado.
     */
    virtual T popBack() = 0;
};

#endif // ILIST_HPP

## Descripción

* ***Consistencia con `ISet`:*** Mantenemos las mismas bases, uso de plantillas (`template <typename T>`), destructor virtual, guarda de inclusión (`#ifndef`) y paso de parámetros complejos por referencia constante (`const T& item`).

* ***Uso de const en métodos de lectura:*** Operaciones como `get()` o `indexOf()` fueron marcadas como `const` al final de la firma. Esto asegura al compilador que llamar a estos métodos no alterará el estado interno de la lista (no modificará los nodos ni el tamaño).

* ***Retorno por valor en `get`, `popFront` y `popBack`:*** Según el diseño en el diagrama UML, estos métodos van retornan `T` (por valor). Esta es una decisión de diseño válida. Significa que al sacar un elemento de la lista, el usuario obtiene una ***copia del dato***.

* ***Nota técnica sobre excepciones***: En C++, si llamas a `popFront()` en una lista vacía, no hay un elemento "por defecto" seguro que devolver. Las implementaciones concretas de esta interfaz (como `DynamicArray` o `SinglyLL`) deberán lanzar una excepción (como `std::out_of_range`) si el usuario intenta acceder a índices inválidos o sacar elementos de una lista vacía.

* ***Retorno de indexOf***: En en diagrama UML se especifica que retorna un `int` y no un `size_t`. Esto es totalmente intencional y correcto. Como `size_t` es un entero sin signo (solo positivos), no puede representar un número negativo. Usar `int` permite devolver `-1` convencionalmente cuando el elemento no se encuentra en la estructura.

# Arreglo Dinámico (`DynamicArray`)
Esta estructura administra la memoria contigua, fundamental para guardar los arreglos internos en ambas técnicas (Encadenamiento y Direccionamiento Abierto).

In [3]:
#ifndef DYNAMICARRAY_HPP
#define DYNAMICARRAY_HPP

#include "IList.hpp"
#include <iostream>
#include <stdexcept> // Necesario para std::out_of_range

/**
 * @brief Implementación de una Lista utilizando un arreglo dinámico.
 *
 * Los elementos se almacenan en bloques de memoria contigua, lo que permite
 * acceso rápido por índice (O(1)). Cuando el arreglo se llena, su capacidad
 * se expande automáticamente.
 *
 * @tparam T Tipo de los elementos que almacenará el arreglo.
 */
template <typename T>
class DynamicArray : public IList<T> {
protected:
    /** @brief Capacidad máxima actual del arreglo antes de necesitar redimensionar. */
    std::size_t capacity;

    /** @brief Puntero al arreglo dinámico subyacente. */
    T* arr;

public:
    /**
     * @brief Constructor por defecto.
     * @param initialCapacity Capacidad inicial del arreglo (por defecto 10).
     */
    DynamicArray(std::size_t initialCapacity = 10) : capacity(initialCapacity) {
        if (capacity == 0) capacity = 1; // Evitamos capacidad 0 para poder multiplicar al redimensionar
        arr = new T[capacity];
        this->size = 0; // Heredado de IList
    }

    /**
     * @brief Destructor. Libera la memoria dinámica del arreglo.
     */
    ~DynamicArray() override {
        delete[] arr;
    }

    /**
     * @brief Obtiene la capacidad actual del arreglo.
     * @return std::size_t La capacidad.
     */
    std::size_t getCapacity() const {
        return capacity;
    }

    /**
     * @brief Redimensiona el arreglo dinámico, duplicando su capacidad.
     */
    void resize() {
        capacity *= 2;
        T* newArr = new T[capacity];
        
        // Copiar los elementos al nuevo arreglo
        for (std::size_t i = 0; i < this->size; ++i) {
            newArr[i] = arr[i];
        }
        
        // Liberar la memoria del arreglo viejo y reasignar el puntero
        delete[] arr;
        arr = newArr;
    }

    // ========================================================================
    // Implementación de los métodos de IList
    // ========================================================================

    void clear() override {
        // No destruimos la memoria, simplemente reiniciamos el contador.
        // Los datos antiguos serán sobrescritos en futuras inserciones.
        this->size = 0;
    }

    void print() const override {
        std::cout << "[";
        for (std::size_t i = 0; i < this->size; ++i) {
            std::cout << arr[i] << (i < this->size - 1 ? ", " : "");
        }
        std::cout << "]\n";
    }

    T get(std::size_t index) const override {
        if (index >= this->size) {
            throw std::out_of_range("Índice fuera de rango en get()");
        }
        return arr[index];
    }

    void set(std::size_t index, const T& item) override {
        if (index >= this->size) {
            throw std::out_of_range("Índice fuera de rango en set()");
        }
        arr[index] = item;
    }

    int indexOf(const T& item) const override {
        for (std::size_t i = 0; i < this->size; ++i) {
            if (arr[i] == item) {
                return static_cast<int>(i);
            }
        }
        return -1;
    }

    void insertAt(std::size_t index, const T& item) override {
        if (index > this->size) {
            throw std::out_of_range("Índice fuera de rango en insertAt()");
        }
        if (this->size == capacity) {
            resize();
        }
        // Desplazar elementos hacia la derecha para hacer espacio
        for (std::size_t i = this->size; i > index; --i) {
            arr[i] = arr[i - 1];
        }
        arr[index] = item;
        this->size++;
    }

    void removeAt(std::size_t index) override {
        if (index >= this->size) {
            throw std::out_of_range("Índice fuera de rango en removeAt()");
        }
        // Desplazar elementos hacia la izquierda para cubrir el hueco
        for (std::size_t i = index; i < this->size - 1; ++i) {
            arr[i] = arr[i + 1];
        }
        this->size--;
    }

    void pushFront(const T& item) override {
        insertAt(0, item);
    }

    void pushBack(const T& item) override {
        if (this->size == capacity) {
            resize();
        }
        arr[this->size++] = item; // Más eficiente que llamar a insertAt(this->size, item)
    }

    T popFront() override {
        if (this->isEmpty()) {
            throw std::out_of_range("La lista está vacía en popFront()");
        }
        T item = arr[0];
        removeAt(0);
        return item;
    }

    T popBack() override {
        if (this->isEmpty()) {
            throw std::out_of_range("La lista está vacía en popBack()");
        }
        // Retornamos el último y simplemente reducimos el tamaño
        return arr[--this->size];
    }
};

#endif // DYNAMICARRAY_HPP

## Descripción

* ***Uso de `this->size`:*** Notarás que cada vez que accedo a la variable size (heredada de IList), uso this->size. En C++, cuando heredas de una clase plantilla (template base class), el compilador no busca automáticamente los nombres heredados por un tema técnico llamado "resolución de nombres en dos fases". Usar `this->` le dice explícitamente al compilador "busca esta variable en la clase base".

* ***Gestión de Memoria (`new[]` y `delete[]`):*** En el constructor alojamos un bloque de memoria contiguo (`new T[capacity]`).Es fundamental usar `delete[]` en el destructor. Como alojamos un arreglo, debemos liberar un arreglo. Usar solo `delete arr`; causaría una fuga de memoria (solo borraría el primer elemento).

* ***La estrategia de `resize()`:*** Cuando el arreglo llega a su capacidad máxima, creamos uno nuevo con el doble de tamaño (`capacity *= 2`). Esta es una técnica estándar conocida como "Amortización de costo". En lugar de crecer de uno en uno (lo que haría que cada `pushBack` fuera costosísimo), duplicar el tamaño garantiza que, en promedio, insertar al final sea una operación de tiempo constante $O(1)$.

* ***Lanzamiento de Excepciones (`std::out_of_range`):*** En métodos como `get`, `removeAt` o `popFront`, si el índice es inválido o la lista está vacía, no podemos hacer nada seguro. Lanzar una excepción de la biblioteca estándar es la forma correcta y robusta en C++17 de comunicar un error fatal que debe ser manejado por quien usa la clase.

* ***Sobreescritura con `override`:*** Se agregó la palabra clave `override` al final de los métodos heredados. Esto es una excelente práctica introducida en C++11/14. Le pide al compilador que verifique que realmente estamos sobrescribiendo un método virtual de la clase base. Si nos equivocamos en una letra del nombre o en los parámetros, el compilador nos dará un error en lugar de crear una función nueva accidentalmente.


# Clase Nodo (`Node`)


In [4]:
#ifndef NODE_HPP
#define NODE_HPP

/**
 * @brief Clase que representa un nodo individual en una estructura enlazada.
 *
 * Cada nodo almacena un dato genérico y un puntero hacia el siguiente
 * nodo en la secuencia.
 *
 * @tparam T Tipo de dato que almacenará el nodo.
 */
template <typename T>
class Node {
public:
    /** @brief Dato almacenado en el nodo. */
    T data;

    /** @brief Puntero al siguiente nodo en la secuencia. */
    Node<T>* next;

    /**
     * @brief Constructor principal del nodo.
     * * @param item El dato a almacenar. Pasado por referencia constante por eficiencia.
     * @param nextNode Puntero al siguiente nodo. Por defecto se inicializa en nullptr.
     */
    explicit Node(const T& item, Node<T>* nextNode = nullptr) 
        : data(item), next(nextNode) {}

    /**
     * @brief Obtiene el dato almacenado en este nodo.
     * @return T Una copia del dato.
     */
    T getData() const {
        return data;
    }

    /**
     * @brief Obtiene el puntero al siguiente nodo.
     * @return Node<T>* El puntero al siguiente nodo.
     */
    Node<T>* getNext() const {
        return next;
    }

    /**
     * @brief Establece el puntero hacia el siguiente nodo.
     * @param node Puntero que se asignará como siguiente.
     */
    void setNext(Node<T>* node) {
        next = node;
    }
};

#endif // NODE_HPP

## Descripcción

* ***Visibilidad Pública (public) vs. Encapsulamiento:*** En el diagrama UML, tanto data como next tienen el prefijo +, lo que indica visibilidad pública. Aunque en programación orientada a objetos pura (cuando existen getters y setters) se suele preferir hacer los atributos privados (-), solo estamos sigueindo el diseño del diagrama UML dejándolos públicos. En C++, es de hecho una práctica muy común y aceptada hacer que los nodos (Node) actúen casi como estructuras (struct) con miembros públicos para facilitar su manipulación por parte de la clase que los contiene (como SinglyLL), evitando el overhead de llamadas a funciones.

* ***Constructor con Lista de Inicialización (`: data(item)`, `next(nextNode)`)***: En lugar de asignar los valores dentro de las llaves del constructor ({}), C++ permite usar listas de inicialización. Esto es más eficiente, especialmente para tipos `T` complejos (como objetos grandes), porque inicializa el objeto directamente en memoria en lugar de crearlo por defecto y luego reasignarlo.

* ***Uso de `nullptr`***: El puntero next se inicializa por defecto con `nullptr`. En C++11 (y mantenido en C++17), `nullptr` es el estándar absoluto para representar un puntero nulo, reemplazando a la antigua macro `NULL` o al entero `0` de C clásico. nullptr es fuertemente tipado (es de tipo `std::nullptr_t`), lo que evita ambigüedades en la sobrecarga de funciones.

* ***Palabra clave `explicit`:*** Se añadió `explicit` al constructor. Esto evita que el compilador realice conversiones implícitas accidentales. Por ejemplo, evita que el compilador trate de convertir automáticamente un valor de tipo `T` en un `Node<T>` en contextos donde se esperaba un nodo pero se pasó un dato suelto.

* ***Retorno por valor en getData()***: Al igual que en la interfaz de la lista, devolvemos una copia de `T`. El método está marcado como `const` porque solo lee el estado del nodo sin modificarlo.

# Lista simplemente enlazada (`SinglyLL`)

In [5]:
#ifndef SINGLYLL_HPP
#define SINGLYLL_HPP

#include "IList.hpp"
#include "Node.hpp"
#include <iostream>
#include <stdexcept>

/**
 * @brief Implementación de una Lista Enlazada Simple (Singly Linked List).
 *
 * Utiliza nodos conectados secuencialmente. Ofrece inserción y eliminación
 * eficiente al principio de la lista (O(1)), pero acceso por índice lineal (O(N)).
 *
 * @tparam T Tipo de los elementos que almacenará la lista.
 */
template <typename T>
class SinglyLL : public IList<T> {
protected:
    /** @brief Puntero al primer nodo de la lista. */
    Node<T>* head;

public:
    /**
     * @brief Constructor por defecto. Inicializa la lista vacía.
     */
    SinglyLL() : head(nullptr) {
        this->size = 0;
    }

    /**
     * @brief Destructor. Libera la memoria de todos los nodos.
     */
    ~SinglyLL() override {
        clear();
    }

    /**
     * @brief Obtiene el puntero al nodo cabeza.
     * @return Node<T>* Puntero al primer nodo.
     */
    Node<T>* getHead() const {
        return head;
    }

    /**
     * @brief Establece un nuevo nodo como cabeza de la lista.
     * @note Este método altera la estructura y podría causar fugas de memoria 
     * si no se maneja con cuidado desde el exterior.
     * @param node Puntero al nuevo nodo cabeza.
     */
    void setHead(Node<T>* node) {
        head = node;
    }

    // ========================================================================
    // Implementación de los métodos de IList
    // ========================================================================

    void clear() override {
        Node<T>* current = head;
        while (current != nullptr) {
            Node<T>* nextNode = current->getNext();
            delete current;
            current = nextNode;
        }
        head = nullptr;
        this->size = 0;
    }

    void print() const override {
        Node<T>* current = head;
        std::cout << "[";
        while (current != nullptr) {
            std::cout << current->getData();
            if (current->getNext() != nullptr) {
                std::cout << " -> ";
            }
            current = current->getNext();
        }
        std::cout << "]\n";
    }

    T get(std::size_t index) const override {
        if (index >= this->size) {
            throw std::out_of_range("Índice fuera de rango en get()");
        }
        Node<T>* current = head;
        for (std::size_t i = 0; i < index; ++i) {
            current = current->getNext();
        }
        return current->getData();
    }

    void set(std::size_t index, const T& item) override {
        if (index >= this->size) {
            throw std::out_of_range("Índice fuera de rango en set()");
        }
        Node<T>* current = head;
        for (std::size_t i = 0; i < index; ++i) {
            current = current->getNext();
        }
        current->data = item; // Acceso directo al dato público del nodo
    }

    int indexOf(const T& item) const override {
        Node<T>* current = head;
        int index = 0;
        while (current != nullptr) {
            if (current->getData() == item) {
                return index;
            }
            current = current->getNext();
            index++;
        }
        return -1; // No encontrado
    }

    void insertAt(std::size_t index, const T& item) override {
        if (index > this->size) {
            throw std::out_of_range("Índice fuera de rango en insertAt()");
        }
        if (index == 0) {
            pushFront(item);
            return;
        }

        // Llegar al nodo ANTERIOR al índice de inserción
        Node<T>* current = head;
        for (std::size_t i = 0; i < index - 1; ++i) {
            current = current->getNext();
        }

        Node<T>* newNode = new Node<T>(item, current->getNext());
        current->setNext(newNode);
        this->size++;
    }

    void removeAt(std::size_t index) override {
        if (index >= this->size) {
            throw std::out_of_range("Índice fuera de rango en removeAt()");
        }
        if (index == 0) {
            popFront();
            return;
        }

        // Llegar al nodo ANTERIOR al que queremos eliminar
        Node<T>* current = head;
        for (std::size_t i = 0; i < index - 1; ++i) {
            current = current->getNext();
        }

        Node<T>* nodeToDelete = current->getNext();
        current->setNext(nodeToDelete->getNext());
        delete nodeToDelete;
        this->size--;
    }

    void pushFront(const T& item) override {
        Node<T>* newNode = new Node<T>(item, head);
        head = newNode;
        this->size++;
    }

    void pushBack(const T& item) override {
        if (this->isEmpty()) {
            pushFront(item);
            return;
        }

        Node<T>* current = head;
        while (current->getNext() != nullptr) {
            current = current->getNext();
        }
        current->setNext(new Node<T>(item));
        this->size++;
    }

    T popFront() override {
        if (this->isEmpty()) {
            throw std::out_of_range("La lista está vacía en popFront()");
        }
        Node<T>* oldHead = head;
        T item = oldHead->getData();
        head = head->getNext();
        delete oldHead;
        this->size--;
        return item;
    }

    T popBack() override {
        if (this->isEmpty()) {
            throw std::out_of_range("La lista está vacía en popBack()");
        }
        if (this->size == 1) {
            return popFront();
        }

        // Llegar al PENÚLTIMO nodo
        Node<T>* current = head;
        while (current->getNext()->getNext() != nullptr) {
            current = current->getNext();
        }

        Node<T>* nodeToDelete = current->getNext();
        T item = nodeToDelete->getData();
        delete nodeToDelete;
        current->setNext(nullptr);
        this->size--;
        return item;
    }
};

#endif // SINGLYLL_HPP

# Descripción 

1. ***Gestión Manual de Memoria en Estructuras Enlazadas***: A diferencia del DynamicArray donde borrábamos un único bloque de memoria con `delete[] arr`;, en una lista enlazada cada nodo fue reservado con su propio `new`. Por lo tanto, el destructor y el método `clear()` deben recorrer la lista nodo por nodo y llamar a `delete` para cada uno. Se utiliza un puntero temporal (`nextNode`) para no perder la referencia al resto de la lista al borrar el nodo actual.

2. ***Costo Computacional (O(N) para el final de la lista)***: Como el diseño del UML no especificaba un puntero tail (cola), operaciones como `pushBack` y `popBack` se ven obligadas a recorrer la lista completa desde head hasta el final utilizando un bucle while. Esto significa que tienen una complejidad temporal de $O(N)$.

3. ***Manejo de Casos Especiales (Bordes)***: En la manipulación de nodos, siempre existen casos borde críticos:

    * Operar en el índice 0 (head).
    * Operar al final de la lista.

    * Operar en una lista vacía.

    Podrás notar que en métodos como `insertAt`, `removeAt` y `popBack`, evaluamos explícitamente el caso del índice 0 o tamaño 1 y los delegamos a `pushFront` / `popFront`. Esto previene fallos de segmentación (segmentation faults) por intentar desreferenciar punteros nulos.

4. ***Precaución con `setHead`:*** El UML pide implementar setHead(node). Esta función permite inyectar un nuevo nodo inicial directamente. Documenté un aviso (`@note`) en el código, ya que en la práctica, sobreescribir head sin haber borrado previamente los nodos existentes provocará una fuga de memoria. Quien use esta función asume la responsabilidad de manejar los nodos previamente enlazados.

# Tabla Hash con Encadenamiento (`ChainedHashTable`).

Dado que en los pasos anteriores diseñamos `IList::get()` para retornar por valor (hace copias) y `SinglyLL` no tiene implementado un constructor de copia personalizado, copiar listas directamente corrompería la memoria (creando punteros colgantes o dobles liberaciones).

Para resolver esto y garantizar una implementación segura en memoria, vamos a realizar una pequeña adaptación al diseño del UML: en lugar de almacenar las listas directamente (`DynamicArray<SinglyLL<T>>`), almacenaremos punteros a las listas (`DynamicArray<SinglyLL<T>*>`).


In [ ]:
#ifndef CHAINEDHASHTABLE_HPP
#define CHAINEDHASHTABLE_HPP

#include "ISet.hpp"
#include "DynamicArray.hpp"
#include "SinglyLL.hpp"
#include <iostream>
#include <functional> // Necesario para std::hash

/**
 * @brief Tabla Hash que maneja colisiones utilizando encadenamiento (Listas Enlazadas).
 *
 * Implementa la interfaz ISet. Los elementos se distribuyen en "cubetas" (buckets)
 * calculados mediante una función Hash. Si hay colisiones, se enlazan en una lista.
 *
 * @tparam T Tipo de los elementos que almacenará el conjunto.
 */
template <typename T>
class ChainedHashTable : public ISet<T> {
private:
    /** * @brief Arreglo dinámico que actúa como tabla. 
     * Almacenamos punteros a SinglyLL para evitar problemas de copia de memoria.
     */
    DynamicArray<SinglyLL<T>*> table;

    /** @brief Tamaño de palabra de la máquina en bits (usualmente 32 o 64). */
    int W;

    /** @brief Número impar aleatorio para dispersar las claves en el hashing MAD. */
    int Z;

    /** @brief Potencia de 2 que determina el tamaño de la tabla (Capacidad = 2^D). */
    int D;

    /**
     * @brief Convierte cualquier tipo de dato T a un entero sin signo.
     * @param item Elemento a convertir.
     * @return unsigned int La representación entera del elemento.
     */
    unsigned int convertToInt(const T& item) const {
        // std::hash es la herramienta estándar en C++11/17 para obtener hashes base
        std::size_t hashValue = std::hash<T>{}(item);
        return static_cast<unsigned int>(hashValue);
    }

    /**
     * @brief Función de dispersión utilizando el método de multiplicación por bits.
     * @param item Elemento a hashear.
     * @return int Índice de la cubeta (bucket) donde pertenece el elemento.
     */
    int hash(const T& item) const {
        unsigned int k = convertToInt(item);
        // Prevenir desbordamiento (overflow) no definido operando con enteros sin signo
        unsigned int result = (static_cast<unsigned int>(Z) * k) >> (W - D);
        return static_cast<int>(result);
    }

public:
    /**
     * @brief Constructor de la tabla hash.
     * @param bitsD Potencia para el tamaño inicial de la tabla (por defecto 4 -> 16 cubetas).
     * @param oddZ Número impar para la función hash (por defecto 33).
     */
    ChainedHashTable(int bitsD = 4, int oddZ = 33) : W(32), Z(oddZ), D(bitsD) {
        int capacity = 1 << D; // 2^D
        for (int i = 0; i < capacity; ++i) {
            // Inicializamos cada cubeta con una lista enlazada vacía
            table.pushBack(new SinglyLL<T>());
        }
        this->size = 0;
    }

    /**
     * @brief Destructor. Libera las listas asignadas dinámicamente.
     */
    ~ChainedHashTable() override {
        int capacity = 1 << D;
        for (int i = 0; i < capacity; ++i) {
            delete table.get(i);
        }
    }

    /**
     * @brief Inserta un elemento en la tabla. 
     * @param item Elemento a insertar.
     * @return true si se insertó, false si ya existía (es un Set).
     */
    bool insert(const T& item) {
        int idx = hash(item);
        SinglyLL<T>* list = table.get(idx);
        
        // Al ser un Set, no permitimos duplicados
        if (list->indexOf(item) != -1) {
            return false;
        }
        
        list->pushFront(item); // O(1) inserción
        this->size++;
        return true;
    }

    // ========================================================================
    // Implementación de los métodos puramente virtuales de ISet
    // ========================================================================

    /**
     * @brief Alias para cumplir con la interfaz ISet, utiliza insert() por debajo.
     */
    bool add(const T& item) override {
        return insert(item);
    }

    bool remove(const T& item) override {
        int idx = hash(item);
        SinglyLL<T>* list = table.get(idx);
        
        int listIdx = list->indexOf(item);
        if (listIdx == -1) {
            return false; // El elemento no está en la tabla
        }
        
        list->removeAt(static_cast<std::size_t>(listIdx));
        this->size--;
        return true;
    }

    bool contains(const T& item) const override {
        int idx = hash(item);
        SinglyLL<T>* list = table.get(idx);
        return list->indexOf(item) != -1;
    }

    void clear() override {
        int capacity = 1 << D;
        for (int i = 0; i < capacity; ++i) {
            table.get(i)->clear();
        }
        this->size = 0;
    }

    void print() const override {
        int capacity = 1 << D;
        std::cout << "--- Tabla Hash ---" << "\n";
        for (int i = 0; i < capacity; ++i) {
            std::cout << "Cubeta " << i << ": ";
            table.get(i)->print();
        }
    }
};

#endif // CHAINEDHASHTABLE_HPP

1. ***Cambio Seguro en la Composición (`DynamicArray<SinglyLL<T>*>`):*** Como mencioné arriba, si usáramos `DynamicArray<SinglyLL<T>>`, cuando extrajéramos una lista usando `table.get(idx)`, obtendríamos una copia por valor. Modificar esa copia (ej. `pushFront`) no modificaría la lista original en la tabla. Peor aún, al destruirse la copia, se borrarían sus nodos, y la lista original quedaría apuntando a basura. Usar punteros resuelve la limitación de la estructura sin requerir reescribir toda la memoria de las clases pasadas, lo cual es una técnica estándar cuando se enlazan contenedores genéricos de C++.

2. ***Método Hashing por Multiplicación y Bits***: El UML provee las variables `W`, `Z` y `D`. Esto nos indica que debemos usar la función de hash basada en desplazamiento de bits (Bitwise hashing):
$$h(k) = (Z \cdot k) \gg (W - D)$$

    Donde $Z$ es un entero impar aleatorio, $k$ es el objeto convertido a entero, $W$ es el número de bits de la palabra (32) y $D$ determina el tamaño de la tabla (el arreglo tendrá $2^D$ posiciones, calculadas en C++ como 1 << D). Esta es una familia de funciones hash extremadamente rápida a nivel de CPU.

3. ***Sobrecarga de `add` vs `insert`:*** El diagrama indicaba que la clase `ChainedHashTable` tiene el método `insert()`, pero implementa la interfaz `ISet`, la cual te obliga a implementar `add()`. Para seguir fielmente el diagrama pero también que el compilador acepte la herencia de `ISet`, se crea el método `insert()` con su propia lógica y he hecho que el método `virtual add()` simplemente lo mande a llamar.

4. ***Seguridad de Desbordamiento (Overflow)***: En C++, el desbordamiento de enteros con signo (int) causa "Comportamiento Indefinido" (Undefined Behavior). Sin embargo, el desbordamiento en enteros sin signo (`unsigned int`) está garantizado por el estándar para dar la vuelta en un comportamiento modular ($mod \ 2^W$). Por eso, en la función hash, realizamos conversiones explícitas (`static_cast`) antes de multiplicar por Z.

5. ***Manejo de Memoria del Constructor y Destructor***: Ya que inicializamos los punteros con memoria dinámica en un bucle dentro del constructor (`new SinglyLL<T>()`), estamos obligados a limpiarlos en el destructor (`delete table.get(i)`). Esto evita filtraciones silenciosas de memoria (Memory Leaks).

## Preguntas 

1. ***¿Tiene sentido hacer rehashing?***

   Absolutamente sí. De hecho, en cualquier implementación de grado de producción (como `std::unordered_set` en C++ o `HashMap` en Java), el rehashing es una característica central. El indicador clave aquí es el *Factor de Carga ($\alpha$)*, que se calcula como $$\alpha = \frac{N}{M}$$

    donde $N$ es el número de elementos y $M$ la cantidad de cubetas. Si no hacemos rehashing, el factor de carga crece infinitamente, y la promesa del acceso en tiempo $O(1)$ se rompe, convirtiéndose en una lenta búsqueda lineal $O(N)$, tal como se comprueba en la prueba de estrés.

3. ***¿Cómo se ve afectado el tiempo al hacer rehashing?***

   El rehashing tiene una doble cara en cuanto al tiempo de ejecución.
    * El impacto inmediato, en el instante exacto en que decides hacer rehashing (por ejemplo, cuando el arreglo se llena al 75%), la operación es muy costosa, de tiempo $O(N)$. Se Debe crear un arreglo nuevo (usualmente del doble de tamaño), y luego recorrer todos los elementos antiguos, recalcular su nuevo índice hash (porque $D$ y la capacidad cambiaron) e insertarlos uno a uno en la nueva tabla.
    * El impacto a largo plazo. Al duplicar el tamaño garantizas que las próximas miles de inserciones vuelvan a ser de tiempo constante $O(1)$. Matemáticamente, si promedias ese gran costo entre todas las inserciones que hiciste, el costo estadístico o tiempo amortizado sigue siendo $O(1)$. Es exactamente el mismo principio que se usa en el método `resize()` de `DynamicArray`.


5. ***¿Sería mejor en una implementación de Direccionamiento Abierto (Linear Probing)?***

    Aquí es donde la diferencia arquitectónica se vuelve crítica. En la tabla con encadenamiento (`ChainedHashTable`), el rehashing es una optimización. Si no lo haces (como en nuestro código), el programa se vuelve lento, pero sigue funcionando porque las listas enlazadas pueden crecer usando memoria dinámica sin límite hasta que se acabe la RAM.En la tabla de direccionamiento abierto (`LinearHashTable`), el rehashing es una obligación absoluta.

# Tabla hash Lineal (`LinearHashTable`)

A diferencia del encadenamiento, el direccionamiento abierto guarda todos los elementos directamente en la tabla principal. Cuando ocurre una colisión, el algoritmo busca la siguiente ranura disponible (índice + 1). Esto introduce el problema de las "cadenas de búsqueda", por lo que necesitaremos un mecanismo especial para borrar elementos sin romper las búsquedas futuras: los famosos estados de ranura (Tombstones/ Marcadores).

In [ ]:
#ifndef LINEARHASHTABLE_HPP
#define LINEARHASHTABLE_HPP

#include "ISet.hpp"
#include "DynamicArray.hpp"
#include <iostream>
#include <functional> // Necesario para std::hash

/**
 * @brief Enum fuertemente tipado para gestionar el estado de cada ranura (slot) en la tabla.
 */
enum class SlotStatus {
    EMPTY,    
    OCCUPIED, 
    DELETED   
};

/**
 * @brief Sobrecarga del operador << para permitir que std::cout imprima un SlotStatus.
 * Esto es necesario porque DynamicArray<SlotStatus> intentará compilar su método virtual print().
 * Se usa 'inline' para evitar errores de definición múltiple si se incluye en varios archivos.
 */
inline std::ostream& operator<<(std::ostream& os, const SlotStatus& status) {
    switch (status) {
        case SlotStatus::EMPTY:    os << "EMPTY"; break;
        case SlotStatus::OCCUPIED: os << "OCCUPIED"; break;
        case SlotStatus::DELETED:  os << "DELETED"; break;
    }
    return os;
}

/**
 * @brief Tabla Hash de direccionamiento abierto mediante Sondeo Lineal (Linear Probing).
 *
 * Mantiene consistencia con ChainedHashTable utilizando el método de 
 * hashing Multiply-Shift (W, Z, D) para la dispersión de claves.
 *
 * @tparam T Tipo de los elementos que almacenará el conjunto.
 */
template <typename T>
class LinearHashTable : public ISet<T> {
private:
    /** @brief Puntero al arreglo dinámico que almacena los datos reales. */
    DynamicArray<T>* table;

    /** @brief Puntero al arreglo dinámico paralelo que almacena el estado de cada ranura. */
    DynamicArray<SlotStatus>* status;

    /** @brief Tamaño de palabra de la máquina en bits (usualmente 32). */
    int W;

    /** @brief Constante multiplicativa impar para dispersar las claves (Constante de Knuth). */
    unsigned int Z;

    /** @brief Potencia de 2 que determina el tamaño de la tabla (Capacidad = 2^D). */
    int D;

    /**
     * @brief Convierte cualquier tipo de dato genérico T a un entero sin signo.
     * @param item Elemento a convertir.
     * @return unsigned int Representación entera base generada por std::hash.
     */
    unsigned int convertToInt(const T& item) const {
        std::size_t hashValue = std::hash<T>{}(item);
        return static_cast<unsigned int>(hashValue);
    }

    /**
     * @brief Función de dispersión consistente usando el método Multiply-Shift.
     * @param item Elemento a evaluar.
     * @return int Índice calculado dentro de los límites de la tabla.
     */
    int hash(const T& item) const {
        unsigned int k = convertToInt(item);
        unsigned int result = (Z * k) >> (W - D);
        return static_cast<int>(result);
    }

public:
    /**
     * @brief Constructor principal. Inicializa la tabla utilizando potencias de 2.
     * @param bitsD Potencia para el tamaño inicial de la tabla (por defecto 4 -> 16 ranuras).
     * @param oddZ Número impar grande para el hash (por defecto la Constante de Knuth).
     */
    LinearHashTable(int bitsD = 4, unsigned int oddZ = 2654435769U) 
        : W(32), Z(oddZ), D(bitsD) {
        
        int capacity = 1 << D; // 2^D
        table = new DynamicArray<T>(capacity);
        status = new DynamicArray<SlotStatus>(capacity);

        // Inicializamos las estructuras subyacentes
        for (int i = 0; i < capacity; ++i) {
            table->pushBack(T{}); // Inserción de valor por defecto según el tipo T
            status->pushBack(SlotStatus::EMPTY);
        }
        this->size = 0;
    }

    /**
     * @brief Destructor. Libera la memoria de los arreglos dinámicos paralelos.
     */
    ~LinearHashTable() override {
        delete table;
        delete status;
    }

    /**
     * @brief Inserta un elemento manejando colisiones mediante sondeo lineal.
     * * Si encuentra una Lápida (DELETED) durante la búsqueda, recicla esa posición
     * para optimizar la memoria y acortar futuras cadenas de búsqueda.
     * * @param item Elemento a insertar.
     * @return true si se insertó con éxito, false si ya existía o la tabla está llena.
     */
    bool insert(const T& item) {
        int capacity = 1 << D;
        // Al no tener rehashing dinámico, prevenimos la inserción si se alcanza el límite
        if (this->size == static_cast<std::size_t>(capacity)) {
            return false;
        }

        int startIdx = hash(item);
        int idx = startIdx;
        int firstDeleted = -1; // Rastreador para reciclar la primera Lápida encontrada

        do {
            SlotStatus currentStatus = status->get(idx);

            if (currentStatus == SlotStatus::EMPTY) {
                // Fin de cadena. Insertamos en el primer DELETED si lo vimos, si no, aquí.
                int targetIdx = (firstDeleted != -1) ? firstDeleted : idx;
                table->set(targetIdx, item);
                status->set(targetIdx, SlotStatus::OCCUPIED);
                this->size++;
                return true;

            } else if (currentStatus == SlotStatus::OCCUPIED) {
                // Comprobamos duplicados (Propiedad Set)
                if (table->get(idx) == item) {
                    return false; 
                }

            } else if (currentStatus == SlotStatus::DELETED) {
                // Registramos la primera Lápida para usarla después si el elemento no es duplicado
                if (firstDeleted == -1) {
                    firstDeleted = idx;
                }
            }

            // Sondeo lineal: avanzamos al siguiente índice de forma circular
            idx = (idx + 1) % capacity;

        } while (idx != startIdx); 

        // Caso extremo: dimos la vuelta entera y estaba llena de Lápidas y ocupados (sin EMPTY)
        if (firstDeleted != -1) {
            table->set(firstDeleted, item);
            status->set(firstDeleted, SlotStatus::OCCUPIED);
            this->size++;
            return true;
        }

        return false;
    }

    // ========================================================================
    // Implementación de los métodos puramente virtuales de ISet
    // ========================================================================

    /**
     * @brief Añade un elemento a la tabla. Interfaz wrapper para insert().
     * @param item Elemento a añadir.
     * @return true si se añadió, false de lo contrario.
     */
    bool add(const T& item) override {
        return insert(item);
    }

    /**
     * @brief Elimina un elemento de la tabla.
     * * No borra el dato físicamente, sino que actualiza su estado a DELETED 
     * para no romper la cadena de búsqueda (Lápida).
     * * @param item Elemento a eliminar.
     * @return true si se encontró y eliminó, false si no existe.
     */
    bool remove(const T& item) override {
        int capacity = 1 << D;
        int startIdx = hash(item);
        int idx = startIdx;

        do {
            SlotStatus currentStatus = status->get(idx);

            if (currentStatus == SlotStatus::EMPTY) {
                return false; // El elemento definitivamente no está en esta cadena
            } else if (currentStatus == SlotStatus::OCCUPIED) {
                if (table->get(idx) == item) {
                    status->set(idx, SlotStatus::DELETED); // Colocamos la Lápida
                    this->size--;
                    return true;
                }
            }
            
            idx = (idx + 1) % capacity;

        } while (idx != startIdx);

        return false;
    }

    /**
     * @brief Verifica si un elemento existe dentro de la tabla.
     * @param item Elemento a buscar.
     * @return true si el elemento está presente, false en caso contrario.
     */
    bool contains(const T& item) const override {
        int capacity = 1 << D;
        int startIdx = hash(item);
        int idx = startIdx;

        do {
            SlotStatus currentStatus = status->get(idx);

            if (currentStatus == SlotStatus::EMPTY) {
                return false; // Fin de cadena, el elemento no existe
            } else if (currentStatus == SlotStatus::OCCUPIED) {
                if (table->get(idx) == item) {
                    return true;
                }
            }
            // Si es DELETED, seguimos sondeando
            idx = (idx + 1) % capacity;

        } while (idx != startIdx);

        return false;
    }

    /**
     * @brief Reinicia la tabla a su estado inicial.
     * Cambia todos los estados a EMPTY de forma lógica (O(N)).
     */
    void clear() override {
        int capacity = 1 << D;
        for (int i = 0; i < capacity; ++i) {
            status->set(i, SlotStatus::EMPTY);
        }
        this->size = 0;
    }

    /**
     * @brief Imprime el estado interno de la tabla en la consola.
     * Muestra las ranuras vacías, las ocupadas y las que contienen Lápidas.
     */
    void print() const override {
        int capacity = 1 << D;
        std::cout << "--- Tabla Hash (Sondeo Lineal) ---\n";
        for (int i = 0; i < capacity; ++i) {
            std::cout << "[" << i << "] ";
            SlotStatus currentStatus = status->get(i);
            
            if (currentStatus == SlotStatus::EMPTY) {
                std::cout << "(Vacio)\n";
            } else if (currentStatus == SlotStatus::DELETED) {
                std::cout << "(Borrado/Lapida)\n";
            } else {
                std::cout << table->get(i) << "\n";
            }
        }
    }
};

#endif // LINEARHASHTABLE_HPP

## Definiciones 
En C++11 se introdujo enum class (conocido como enumeración fuertemente tipada o scoped enum), que viene a mejorar el enum clásico de C.

```
enum class SlotStatus {
    EMPTY,
    OCCUPIED,
    DELETED
};
```

* ***Seguridad de tipos (Type Safety)***: Un enum class no se convierte automáticamente a números enteros. Si intentas hacer `if (status == 1)`, el compilador te dará un error. Te obliga a ser explícito y usar `SlotStatus::OCCUPIED`, lo que evita muchísimos errores lógicos.

* ***Espacio de nombres (Scoping)***: Los valores `EMPTY`, `OCCUPIED` y `DELETED` viven dentro de `SlotStatus`. Si en otra parte de tu código creas otra variable u otro enum que también tenga la palabra `EMPTY`, no chocarán entre sí, porque para referirte a este debes escribir obligatoriamente `SlotStatus::EMPTY`.

En una tabla hash con  (Lineal), cuando hay una colisión, el nuevo elemento simplemente avanza a la siguiente casilla de la derecha hasta encontrar un espacio.

Esto crea "cadenas de búsqueda". El problema grave ocurre cuando quieres borrar un elemento que está en medio de esa cadena.

Imagina este escenario paso a paso. Tenemos una tabla donde la función hash para las palabras "Ana", "Beto" y "Carlos" devuelve el mismo índice: 2.

1. ***Paso A: Inserción***

    + Insertamos "Ana": Va al índice 2. Estado: OCCUPIED.
    + Insertamos "Beto": El índice 2 está ocupado. Salta al 3. Estado: OCCUPIED.
    + Insertamos "Carlos": Los índices 2 y 3 están ocupados. Salta al 4. Estado: OCCUPIED.

    Nuestra tabla se ve así:

    `[2] Ana (OCCUPIED)`
   
    `[3] Beto (OCCUPIED)`

    `[4] Carlos (OCCUPIED)`

    `[5] (EMPTY)`

2. ***Paso B: El problema de borrar MAL***
    Imagina que ahora queremos borrar a "Beto". Si simplemente marcamos el índice 3 como EMPTY, la tabla queda así:

    `[2] Ana (OCCUPIED)`

    `[3] (EMPTY)`

    `[4] Carlos (OCCUPIED)`

    Ahora, nos preguntamos: "¿la tabla contiene a Carlos?".
    El algoritmo calcula el hash de "Carlos" (que es 2), va al índice 2, ve a "Ana", y salta al 3. Ve que el 3 está `EMPTY`. La regla del algoritmo dice: "Si encuentras un `EMPTY`, significa que llegaste al final de la cadena y el elemento no existe". El algoritmo devuelve false. ¡Acabamos de perder a Carlos virtualmente!

3. ***Paso C: La solución con `SlotStatus::DELETED` (Tombstones)***
En lugar de marcar el espacio de "Beto" como` EMPTY`, lo marcamos como `DELETED`.

    `[2] Ana (OCCUPIED)`

    `[3] (DELETED)`

    `[4] Carlos (OCCUPIED)`

    `[5] (EMPTY)`

    Ahora, cuando busquemos a "Carlos":

    + Vamos al índice 2 (Ana). No es Carlos, saltamos.
    + Vamos al índice 3 (`DELETED`). La regla ahora dice: "Aquí hubo un dato. La cadena podría continuar. Sigue buscando". Saltamos.
    + Vamos al índice 4. ¡Encontramos a Carlos! Devolvemos true.

    Nuestros estado indican:
    + `EMPTY`: Detiene las búsquedas. (Nadie insertó nada aquí nunca).
    + `OCCUPIED`: Revisa este dato. Hay un elemento válido viviendo aquí.
    + `DELETED`: Sigue buscando. Hubo un dato aquí, pero fue borrado. No detengas la búsqueda porque podría haber elementos colisionados más adelante, pero sí puedes usar este espacio para insertar un elemento nuevo si lo necesitas.

## ¿Cómo funciona exactamente el Sondeo Lineal (Búsqueda Lineal)?

El corazón de la estrategia de direccionamiento abierto está en la instrucción `idx = (idx + 1) % capacity;`. Esta simple fórmula matemática es la que define el Sondeo Lineal (Linear Probing).

Tomemos como ejemplo el método `contains(item)`. Así es como el algoritmo navega por el arreglo

* El Punto de Partida (`startIdx`): Primero, pasamos el item por la función hash. Esto nos da el índice ideal donde debería estar el elemento. Guardamos este valor en startIdx.

* El Bucle do-while: Entramos a un bucle que inspecciona la casilla actual (`idx`).

* Las 3 Evaluaciones:

  + Si el estado es `EMPTY`: Significa que hemos llegado al final de la "cadena de colisiones". Si el elemento que buscamos existiera, habría sido insertado antes de este espacio vacío. Por lo tanto, la búsqueda termina inmediatamente y devolvemos false.

  + Si el estado es `OCCUPIED`: Comparamos el valor de la tabla con nuestro item. Si coinciden retorna `true`. Si no coinciden, es una colisión de otro elemento, por lo que ignoramos y nos preparamos para avanzar.

  + Si el estado es `DELETED`: Ignoramos y nos preparamos para avanzar, ya que el elemento podría estar más adelante.

  + El Salto (`idx = (idx + 1) % capacity`): Esta es la búsqueda lineal. Le sumamos 1 al índice actual para revisar la casilla contigua a la derecha.

  + El uso del operador módulo (%): Si estamos en el último índice del arreglo (ej. capacidad 16, índice 15) y sumamos 1, obtenemos 16. Como no existe el índice 16, 16 % 16 = 0. Esto crea un efecto de ciclo, regresando al principio del arreglo para seguir buscando.

  + La Condición de Parada de Emergencia (`while (idx != startIdx)`): Si la tabla está completamente llena de elementos y lápidas (sin un solo `EMPTY`), la búsqueda daría vueltas infinitamente. Esta condición asegura que si el índice de búsqueda da la vuelta completa y vuelve a tocar el `startIdx` original, el bucle se rompe y la función asume que el elemento no existe.


## Decisiones de Diseño TécnicoEstructura de Arreglos (SoA) vs. Arreglo de Estructuras (AoS):

El diagrama UML dictó una arquitectura muy específica:
- -`table: DynamicArray~T~*`
- -`status: DynamicArray~SlotStatus~*`

Esta decisión arquitectónica se conoce como SoA (Structure of Arrays). En lugar de crear un objeto que contenga el dato y su estado juntos, usamos dos arreglos paralelos distintos. Desde el punto de vista de optimización en C++, esto es excelente para la memoria caché de la CPU. Cuando el algoritmo itera buscando estados `EMPTY` o `DELETED`, la CPU solo carga el arreglo status en la caché L1, permitiendo una iteración ultra rápida sin contaminar la caché con los datos pesados de tipo `T` hasta que realmente los necesitamos.


***Re-cálculo Dinámico de Capacidad (1 << D):*** En los métodos como `insert` y `remove`, en lugar de usar una variable estática `capacity`, recalculamos `int capacity = 1 << D;`. Esto es una optimización matemática usando desplazamiento de bits (bitshifting). Calcular $2^D$ mediante $1 << D$ toma un solo ciclo de reloj del procesador. Es tan rápido que es preferible recalcularlo localmente a nivel de método que gastar memoria almacenando una copia extra de la variable. 

***Ausencia de Rehashing Automático:*** A diferencia de la clase `DynamicArray` donde programamos el método `resize()`, en esta tabla hash el método `insert()` retorna `false` si el tamaño alcanza la capacidad. Esta es una decisión deliberada y descrita en el esquema UML, el cual no incluye métodos privados para reconstruir la tabla. En C++, implementar un resize en direccionamiento abierto requiere crear una tabla completamente nueva del doble de tamaño, re-calcular los hashes de cada elemento vivo e insertarlos uno por uno. Quien instancie esta clase debe usar un valor de $D$ lo suficientemente grande desde el principio. 

***Seguridad contra Fugas de Memoria en la Limpieza (`clear()`):*** El método `clear()` tiene una decisión de diseño crítica, no borra ni elimina los elementos de tipo `T` en `table`. Simplemente marca todos los estados del arreglo status como `EMPTY` y reinicia el tamaño a 0. Esto es una operación increíblemente rápida de $O(N)$ basada solo en banderas. Los datos antiguos en `table` permanecen como "basura" en la memoria, pero es seguro, ya que serán sobrescritos automáticamente en futuras inserciones cuando el algoritmo los detecte como `EMPTY`.